# Chapter 17 — Blur Filters

*Companion notebook for* **Foundations of Computer Vision** *(Torralba, Isola, Freeman), Ch. 17 — [visionbook.mit.edu](https://visionbook.mit.edu/blurring_2.html).*

Blur filters are **low-pass** linear filters: they attenuate high spatial frequencies (fine detail and noise) while preserving the low-frequency structure of an image. This notebook builds the three filter families the chapter develops — the **box filter**, the **Gaussian filter**, and the **binomial filter** — implements each as a convolution, and reproduces the chapter's figures from the underlying math.

The four photo-based figures (17.1, 17.4, 17.5, 17.8) use the **book's own example images** — its noisy stop sign, zebra, Lincoln block portrait, and checkerboard boat — loaded live from `visionbook.mit.edu` so the demonstrations match the textbook exactly. The remaining figures use a `scikit-image` sample.

In [ ]:
import io, urllib.request
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from skimage import data, img_as_float

torch.manual_seed(0)
np.random.seed(0)
torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "image.cmap": "gray",
    "image.interpolation": "nearest",
    "axes.grid": False,
})

# Grayscale sample image, float32 in [0, 1] — our stand-in for the book photos.
IMG = torch.from_numpy(img_as_float(data.camera())).float()   # 512x512 'cameraman'
H, W = IMG.shape

# The four photo-based figures (17.1, 17.4, 17.5, 17.8) use the BOOK's own
# example images, loaded straight from visionbook.mit.edu so the demonstrations
# match the textbook exactly. They are referenced online, not redistributed here.
BOOK_FIG = 'https://visionbook.mit.edu/figures'


def load_book_gray(url):
    """Fetch a book figure and return it as a float32 grayscale tensor in [0, 1]."""
    with urllib.request.urlopen(url, timeout=30) as r:
        im = Image.open(io.BytesIO(r.read())).convert('L')
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)


def load_book_rgb(url):
    """Fetch a book figure as a float32 RGB tensor (H, W, 3) in [0, 1]."""
    with urllib.request.urlopen(url, timeout=30) as r:
        im = Image.open(io.BytesIO(r.read())).convert('RGB')
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)


def conv2d(image, kernel, mode='reflect'):
    """2D convolution, "same" size, reflect-padded.

    Kernels here are symmetric, so convolution == cross-correlation; we still
    flip for correctness. `kernel` is expected to be DC-normalised (sums to 1)
    when a brightness-preserving blur is wanted.
    """
    k = torch.as_tensor(kernel, dtype=torch.float32).flip(0).flip(1)
    kh, kw = k.shape
    x = F.pad(image[None, None], (kw // 2, kw // 2, kh // 2, kh // 2), mode=mode)
    return F.conv2d(x, k[None, None])[0, 0]


def conv2d_rgb(image, kernel, mode='reflect'):
    """Apply the same 2D kernel to each channel of an (H, W, 3) colour image."""
    return torch.stack([conv2d(image[..., c], kernel, mode) for c in range(3)], dim=-1)


def show(panels, titles, figsize=None, cmaps=None):
    """Render a row of images with shared grayscale scaling by default."""
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=figsize or (3.4 * n, 3.6))
    if n == 1:
        axes = [axes]
    for ax, im, t in zip(axes, panels, titles):
        arr = im.detach().cpu().numpy() if torch.is_tensor(im) else im
        if arr.ndim == 3:               # (H, W, 3) colour
            ax.imshow(np.clip(arr, 0, 1))
        else:                            # (H, W) grayscale
            ax.imshow(arr, vmin=0, vmax=1)
        ax.set_title(t, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

## 17.1 — Why blur? Noise removal versus detail loss

A blur filter replaces each pixel with a **weighted average of its neighbours**. Averaging suppresses zero-mean noise (the fluctuations cancel) but also smears genuine high-frequency detail — the central trade-off of the whole chapter. Figure 17.1 makes the trade-off visible: additive noise is largely gone after a $5\times5$ average, at the cost of sharpness.

In [ ]:
# Figure 17.1 — the book's noisy 'stop' image (colour), denoised by a box average.
# The image has 3 channels; the box is applied to each (R, G, B) independently.
stop_noisy = load_book_rgb(f'{BOOK_FIG}/blur_filters/stop_256_noise_3.jpg')
avg5 = torch.ones(5, 5) / 25.0            # normalised 5x5 box (DC gain = 1)
denoised = conv2d_rgb(stop_noisy, avg5)

show([stop_noisy, denoised],
     ['(a) noisy input (book stop_256)', '(b) 5x5 box average'])
print('input  std (noisy):', stop_noisy.std().item())
print('output std (blurred):', denoised.std().item(),
      '  # lower = high-frequency noise attenuated')

## 17.2 — The box filter

The 2D **box** (moving-average) kernel keeps a constant weight inside a rectangular window and zero outside:

$$\mathrm{box}_{N,M}[n,m] = \begin{cases}1 & -N \le n \le N,\ -M \le m \le M\\ 0 & \text{otherwise}\end{cases}$$

To keep average brightness unchanged the kernel must have **DC gain 1**, i.e. its coefficients sum to 1, so we divide by $(2N+1)(2M+1)$.

The box is **separable**: $\mathrm{box}_{N,M} = \mathrm{box}_N^{\,x} * \mathrm{box}_M^{\,y}$ — a full-window rectangle equals a horizontal bar convolved with a vertical bar. Choosing $N=0$ or $M=0$ blurs along a single axis, as the middle and right panels of Figure 17.2 show.

In [ ]:
# Figure 17.2 — square vs horizontal vs vertical box blur.
def box_kernel(N, M):
    return torch.ones(2 * N + 1, 2 * M + 1) / ((2 * N + 1) * (2 * M + 1))

square = conv2d(IMG, box_kernel(6, 6))    # 13x13 window
horiz  = conv2d(IMG, box_kernel(0, 12))   # 1x25  -> blurs horizontally
vert   = conv2d(IMG, box_kernel(12, 0))   # 25x1  -> blurs vertically

show([IMG, square, horiz, vert],
     ['input', '13x13 square', 'horizontal (1x25)', 'vertical (25x1)'])

# Separability check: 2D box == 1D-x then 1D-y.
sep = conv2d(conv2d(IMG, box_kernel(0, 6)), box_kernel(6, 0))
print('max |2D-box - separable|:', (square - sep).abs().max().item())

### 17.2.1 — The box filter's frequency response is not monotonic

The discrete-time Fourier transform of a length-$L$ box is a **Dirichlet (aliased-sinc) kernel**. Because a sinc oscillates, the box's frequency response has **side lobes**: some high frequencies are passed with *more* gain than lower ones, and the sign flips lobe-to-lobe. A good low-pass filter should instead fall off monotonically — the motivation for the Gaussian and binomial filters below.

In [ ]:
# Figure 17.3 — the book's box_1 = [1, 1, 1] kernel and its (rippled) Fourier
# magnitude. |Box_1(w)| = |1 + 2 cos(w)| dips to 0 at f=1/3 then rises again to a
# side lobe at f=1/2 — non-monotonic, unlike the Gaussian/binomial.
box1d = np.array([1.0, 1.0, 1.0])        # box_1 (N=1), unnormalised
NFFT = 512
H_box = np.fft.fftshift(np.fft.fft(box1d, NFFT))
freq  = np.fft.fftshift(np.fft.fftfreq(NFFT))

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.4))
a.stem([-1, 0, 1], box1d)
a.set_title('(a) box_1 kernel  [1, 1, 1]'); a.set_xlabel('n'); a.set_ylabel('weight')
a.set_xlim(-5, 5)
b.plot(freq, np.abs(H_box))
b.set_title('(b) |DFT| — dip at f=1/3, side lobe at f=1/2')
b.set_xlabel('normalised frequency'); b.set_ylabel('magnitude')
plt.tight_layout(); plt.show()

## 17.3 — The Gaussian filter

The **Gaussian** is the canonical isotropic blur. Continuous form:

$$g(x;\sigma) = \frac{1}{\sqrt{2\pi\sigma^2}}\,e^{-x^2/(2\sigma^2)},\qquad g(x,y;\sigma) = \frac{1}{2\pi\sigma^2}\,e^{-(x^2+y^2)/(2\sigma^2)}.$$

We **discretise** by sampling $e^{-(n^2+m^2)/(2\sigma^2)}$ on the integer grid and renormalising to unit sum. Samples beyond $\pm 3\sigma$ are negligible, so a radius of $\lceil 3\sigma\rceil$ suffices.

The Gaussian is the **only** circularly symmetric kernel that is also **separable**: $g(x,y) = g(x)\,g(y)$. Filtering with two 1D passes costs $O(2N)$ per pixel instead of $O(N^2)$.

In [ ]:
def gaussian_1d(sigma, radius=None):
    if radius is None:
        radius = int(np.ceil(3 * sigma))
    x = torch.arange(-radius, radius + 1, dtype=torch.float32)
    k = torch.exp(-x**2 / (2 * sigma**2))
    return k / k.sum()

def gaussian_2d(sigma, radius=None):
    k = gaussian_1d(sigma, radius)
    K = torch.outer(k, k)
    return K / K.sum()

# Separability check at sigma = 4.
g1 = gaussian_1d(4.0)
full  = conv2d(IMG, gaussian_2d(4.0))
cascade = conv2d(conv2d(IMG, g1[None, :]), g1[:, None])   # 1D-x then 1D-y
print('radius at sigma=4 :', (len(g1) - 1) // 2)
print('max |2D - cascade|:', (full - cascade).abs().max().item())

In [ ]:
# Figure 17.4 — progressive Gaussian blur of the book's zebra.
# The book publishes the zebra already blurred at sigma=2; we take that as the
# base and push it to sigma=4 and sigma=8 using the composition rule
# sigma_total^2 = sigma_base^2 + sigma_add^2 (see 17.3.1) -- so this figure also
# demonstrates that identity on a real image.
zebra2 = load_book_gray(f'{BOOK_FIG}/spatial_filters/gausian_zebra_c_2.jpg')  # sigma=2
z4 = conv2d(zebra2, gaussian_2d((4.0**2 - 2.0**2) ** 0.5))   # add sqrt(12) -> total 4
z8 = conv2d(zebra2, gaussian_2d((8.0**2 - 2.0**2) ** 0.5))   # add sqrt(60) -> total 8
show([zebra2, z4, z8],
     ['book zebra, sigma=2', 'blurred to sigma=4', 'blurred to sigma=8'])

### 17.3.1 — Properties of the Gaussian

1. **Its Fourier transform is another Gaussian**, $G(\omega;\sigma)=e^{-\omega^2\sigma^2/2}$, which is **monotonically decreasing** — no side lobes (contrast Figure 17.3b). A wider Gaussian in space is a *narrower* one in frequency.
2. **Composition adds variances:** $g(\sigma_1)*g(\sigma_2)=g(\sigma_3)$ with $\sigma_3^2=\sigma_1^2+\sigma_2^2$. Blurring twice is blurring once by a larger $\sigma$.

Both properties hold *exactly* only in the continuous case; the sampled kernel satisfies them to a close approximation.

In [ ]:
# (left) monotonic Gaussian frequency response vs the rippled box;
# (right) numerical check that g(s1) * g(s2) == g(sqrt(s1^2+s2^2)).
g = gaussian_1d(3.0, radius=32).numpy()
box = (np.ones(9) / 9)
NFFT = 512; freq = np.fft.fftshift(np.fft.fftfreq(NFFT))
Hg  = np.abs(np.fft.fftshift(np.fft.fft(g,   NFFT)))
Hb  = np.abs(np.fft.fftshift(np.fft.fft(box, NFFT)))

s1, s2 = 3.0, 4.0
lhs = np.convolve(gaussian_1d(s1, 40).numpy(), gaussian_1d(s2, 40).numpy())
s3 = np.sqrt(s1**2 + s2**2)
r = (len(lhs) - 1) // 2
rhs = gaussian_1d(s3, r).numpy()

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.4))
a.plot(freq, Hg / Hg.max(), label='Gaussian (monotonic)')
a.plot(freq, Hb / Hb.max(), label='box (ripples)', alpha=0.8)
a.set_title('(a) frequency response'); a.set_xlabel('normalised frequency')
a.legend(fontsize=8)
b.plot(lhs, label='g(s1) * g(s2)', lw=3, alpha=0.5)
b.plot(np.arange(len(rhs)) + (len(lhs) - len(rhs)) // 2, rhs,
       '--', label=f'g(sqrt(s1^2+s2^2)) = g({s3:.1f})')
b.set_title('(b) variances add'); b.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('len(lhs), len(rhs):', len(lhs), len(rhs))
print('max |lhs - rhs|   :', np.abs(lhs - rhs).max())

### 17.3.2 — Blur as a perceptual low-pass: the block portrait

Harmon & Julesz's famous demonstration: a face quantised into coarse blocks is hard to read because the **block edges inject high-frequency energy** that the visual system latches onto. Low-pass filtering (Gaussian blur — or simply squinting) removes those spurious high frequencies and the face re-emerges. Below we take the book's own blocky Lincoln portrait and blur it.

In [ ]:
# Figure 17.5 — the book's blocky Lincoln (Harmon & Julesz 1971); blur reveals
# the face by removing the block-edge high frequencies.
lincoln = load_book_gray(f'{BOOK_FIG}/blur_filters/Jules_Lincoln_1971.jpg')
revealed = conv2d(lincoln, gaussian_2d(lincoln.shape[0] / 45.0))   # sigma ~ one block
show([lincoln, revealed],
     ['(a) block portrait (Harmon-Julesz 1971)', '(b) Gaussian blur -> face emerges'])

## 17.4 — Binomial filters

A **binomial filter** is what you get by convolving the elementary two-tap averager $[1,1]$ with itself $n$ times. The coefficients are the $n$-th row of **Pascal's triangle**:

$$b_1=[1,1],\quad b_2=[1,2,1],\quad b_3=[1,3,3,1],\quad\dots$$

Key facts (discrete analogues of the Gaussian's):

- **DC gain** $=\sum b_n = 2^n$, so normalise by $2^n$.
- **Variance** $\sigma_n^2 = n/4$.
- **Composition:** $b_n * b_m = b_{n+m}$ and $\sigma_n^2+\sigma_m^2=\sigma_{n+m}^2$.
- The Fourier magnitude $B_{2n}(u)=(2+2\cos(2\pi u/N))^n$ is **zero-phase and monotonic** — a discrete filter with no side lobes.

In [ ]:
# Figure 17.7 — the 1D binomial [1,2,1] and its monotonic Fourier magnitude.
def binomial_1d(n):
    b = np.array([1.0])
    for _ in range(n):
        b = np.convolve(b, [1.0, 1.0])   # repeated [1,1] convolution
    return b

b2 = binomial_1d(2)                       # [1, 2, 1]
var = np.sum(np.arange(-(len(b2)//2), len(b2)//2 + 1)**2 * b2) / b2.sum()
print('b2         :', b2, ' DC gain =', int(b2.sum()), '= 2^2')
print('variance   :', var, ' (expected n/4 = 0.5)')

NFFT = 512; freq = np.fft.fftshift(np.fft.fftfreq(NFFT))
Hb2 = np.abs(np.fft.fftshift(np.fft.fft(b2, NFFT)))
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.4))
a.stem([-1, 0, 1], b2); a.set_title('(a) binomial b2 = [1, 2, 1]')
a.set_xlabel('n'); a.set_ylabel('weight')
b.plot(freq, Hb2); b.set_title('(b) |DFT| — monotonic, no side lobes')
b.set_xlabel('normalised frequency')
plt.tight_layout(); plt.show()

### 17.4.1 — 2D binomial filters and perfect cancellation of the checkerboard

By separability the 2D binomial is the outer product $b_{2,2}=[1,2,1]^\top[1,2,1]$, i.e.

$$\tfrac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}.$$

The highest representable frequency is the alternating wave $[\dots,1,-1,1,-1,\dots]$. Convolving it with $[1,2,1]/4$ gives $(1-2+1)/4 = 0$ — the binomial **annihilates the checkerboard exactly**. The box $[1,1,1]/3$ leaves a residual $(1-1+1)/3 = 1/3$. Figure 17.8 shows this on an image corrupted by a checkerboard pattern.

In [ ]:
# Figure 17.8 — checkerboard cancellation on the book's boat.
# The book displays the checkerboard as 8x8-pixel squares (an 8x upscaled view);
# on the actual sampling grid it is the 1-pixel Nyquist wave [1,-1,...]. We take
# the book's boat, add that 1-pixel checkerboard, then filter: the 3x3 box leaves
# a 1/3 residual, while the 3x3 binomial [1,2,1]^2/16 annihilates it exactly.
boat = load_book_rgb(f'{BOOK_FIG}/blur_filters/boat_d_binomial.jpg')   # book's boat (colour)
boat = F.avg_pool2d(boat.permute(2, 0, 1)[None], 4)[0].permute(1, 2, 0)   # -> 256px clean base
Hb, Wb = boat.shape[:2]
yy, xx = torch.meshgrid(torch.arange(Hb), torch.arange(Wb), indexing='ij')
checker = ((xx + yy) % 2 == 0).float() * 2 - 1         # +-1 alternating (highest freq)
corrupt = (boat + 0.4 * checker[..., None]).clamp(0, 1)   # same checker added to R, G, B

box3 = torch.ones(3, 3) / 9.0
b = torch.tensor(binomial_1d(2), dtype=torch.float32)
bin2d = torch.outer(b, b); bin2d = bin2d / bin2d.sum()   # [1,2,1]^2 / 16

out_box = conv2d_rgb(corrupt, box3)
out_bin = conv2d_rgb(corrupt, bin2d)
show([corrupt, out_box, out_bin],
     ['(a) boat + checkerboard', '(b) 3x3 box (residual)', '(c) 3x3 binomial (clean)'])

print('recovered-vs-clean error — box     :', (out_box - boat).abs().mean().item())
print('recovered-vs-clean error — binomial:', (out_bin - boat).abs().mean().item())
print('residual checkerboard    — binomial:', (out_bin * checker[..., None]).mean().abs().item())

### 17.4.2 — Binomials converge to the Gaussian

Repeatedly convolving $[1,1]$ is repeated averaging, so by the **central limit theorem** the normalised binomial $b_n/2^n$ approaches a Gaussian of variance $n/4$ as $n$ grows. This is why small binomials (e.g. $[1,2,1]$, $[1,4,6,4,1]$) are the standard cheap, integer-arithmetic Gaussian approximations used in image pyramids.

In [ ]:
# Overlay the normalised binomial b_n against the Gaussian of matching variance.
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, n in zip(axes, [2, 8, 32]):
    b = binomial_1d(n); b = b / b.sum()
    idx = np.arange(-(len(b) // 2), len(b) // 2 + 1)
    sigma = np.sqrt(n / 4)
    g = np.exp(-idx**2 / (2 * sigma**2)); g = g / g.sum()
    ax.stem(idx, b, linefmt='C0-', markerfmt='C0o', basefmt=' ', label='binomial/2^n')
    ax.plot(idx, g, 'C3--', lw=2, label=f'Gaussian sigma={sigma:.2f}')
    ax.set_title(f'n = {n}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 17.5 — Concluding remarks

Three blur filters, one theme — a **low-pass average**:

| Filter | Kernel | Frequency response | Notes |
|---|---|---|---|
| **Box** | uniform window | sinc — **side lobes** | cheapest; separable; artefacts |
| **Gaussian** | $e^{-x^2/2\sigma^2}$ | Gaussian — **monotonic** | isotropic; only separable circular kernel; variances add |
| **Binomial** | Pascal's row $/2^n$ | $(2+2\cos)^n$ — **monotonic** | integer Gaussian approx; $b_n*b_m=b_{n+m}$; cancels the checkerboard |

The box is fast but its ripples pass spurious high frequencies. The Gaussian is the ideal isotropic low-pass and composes cleanly under $\sigma^2$ addition. The binomial is the practical, integer-arithmetic bridge — a discrete filter that keeps the Gaussian's good behaviour and, by the central limit theorem, *becomes* a Gaussian in the limit. These are the building blocks for **downsampling, upsampling, and image pyramids** in the chapters that follow.